In [1]:
import random
from itertools import chain
import pandas as pd
from teamwork import teamwork as tw
import networkx as nx
from itertools import combinations  
import requests 
import numpy as np
from datetime import datetime, date, timedelta

random.seed(1)

class DataGenerator:
    def __init__(self,num_docs=10,num_patients=100,min_notes_per_discharge=5,max_notes_per_discharge=10,
                 max_stay_per_patient=6,start=None,end=None):
        self.MIN_NOTES_PER_DIS = min_notes_per_discharge
        self.MAX_NOTES_PER_DIS = max_notes_per_discharge
        self.MAX_STAY_PER_PATIENT = max_stay_per_patient     
        self.drs = [name for name in self._get_random_names(num_docs)]
        self.drs_non_collab = [name for name in self._get_random_names(num_docs)]
        self.patients = [name for name in self._get_random_names(num_patients, "patient")]    
        self.end = np.datetime64('now') if end == None else np.datetime64(end)
        self.start = self.end - np.timedelta64(365,'D') if start == None else np.datetime64(start)
        self._create_date_range()
        self._generate_data()

    def _get_random_names(self,count, prefix="prov"):
        names = [f"{prefix}{i+1}" for i in range(0,count)]
        return names

    def _get_date(self):
        return random.choice(self.date_range) + np.timedelta64(random.randint(1,86400),'s')

    def _generate_notes_for_discharge(self, discharge):
#         disposition = discharge[4]
#         dr_list = self.drs if disposition < 1 else self.drs_non_collab
        dr_list = random.choices(self.drs,k=random.randint(6,10))
        notes = [(discharge[0], random.choice(dr_list), discharge[2] + self._get_random_timedelta()) 
                 for _ in range(0, random.randint(self.MIN_NOTES_PER_DIS,self.MAX_NOTES_PER_DIS))]
        return notes

    def _create_date_range(self):
        dr = np.arange(self.start, self.end, np.timedelta64(1, 'D'))
        self.date_range = dr
#         self.date_range = [date for date in dr.astype(object)]

    def _generate_notes(self,discharges):
        for d in discharges:
            yield self._generate_notes_for_discharge(d)

    def _get_random_timedelta(self):
        return np.timedelta64(random.randint(1,self.MAX_STAY_PER_PATIENT - 1), 'D') + np.timedelta64(random.randint(1,86400),'s')
    
    def _flip(self):
        return 0 if random.random() > .1 else 1

    def _generate_data(self):  
        discharges = [(i, p, d := self._get_date(), d + np.timedelta64(self.MAX_STAY_PER_PATIENT - 1,'D') + np.timedelta64(random.randint(1,86400),'s'), self._flip(), random.randint(65,80))
            for i,p in enumerate(self.patients)]

        notes = list(chain.from_iterable(self._generate_notes(discharges)))
        
        dis_columns = ['id', 'patient', 'arrive_date', 'discharge_date', 'disposition', 'age']

        self.dis_df = pd.DataFrame(discharges, columns=dis_columns)

        note_columns=['id', 'dr', 'date']
        
        self.note_df = pd.DataFrame(notes, columns=note_columns)

        self.merged = self.note_df.merge(self.dis_df, on=['id'])


    
# if __name__ == "__main__":
#     data = DataGenerator(num_docs=30, num_patients=70, max_stay_per_patient=7,min_notes_per_discharge=15,max_notes_per_discharge=18)
#     print(data.note_df.head())
#     print(data.note_df.shape)
#     print(data.dis_df.head())
#     print(data.dis_df.shape)
    
#     data.note_df.to_csv('../data/notes_w_datetime.csv', index=False)
#     data.dis_df.to_csv('../data/discharges_w_datetime.csv', index=False)

In [2]:
start_date = pd.Timestamp('2023-01-01')
end_date = pd.Timestamp('2024-01-01')

data = DataGenerator(num_docs=20, num_patients=50, max_stay_per_patient=7,min_notes_per_discharge=15,max_notes_per_discharge=18, start=start_date, end=end_date)
print(data.note_df.head())
print(data.note_df.shape)
print(data.dis_df.head())
print(data.dis_df.shape)

   id      dr                date
0   0  prov13 2023-03-16 16:47:31
1   0  prov16 2023-03-13 22:08:57
2   0   prov6 2023-03-12 10:34:05
3   0  prov13 2023-03-13 09:08:10
4   0  prov16 2023-03-16 18:06:40
(833, 3)
   id   patient         arrive_date      discharge_date  disposition  age
0   0  patient1 2023-03-10 20:43:27 2023-03-16 23:01:19            0   80
1   1  patient2 2023-08-19 17:11:39 2023-08-26 16:55:05            0   71
2   2  patient3 2023-02-18 17:45:45 2023-02-24 18:47:41            0   77
3   3  patient4 2023-08-10 22:06:59 2023-08-16 22:11:36            0   73
4   4  patient5 2023-04-28 21:31:24 2023-05-05 01:14:44            0   65
(50, 6)


In [3]:
data.merged

,id,dr,date,patient,arrive_date,discharge_date,disposition,age
0,0,prov13,2023-03-16 16:47:31,patient1,2023-03-10 20:43:27,2023-03-16 23:01:19,0,80
1,0,prov16,2023-03-13 22:08:57,patient1,2023-03-10 20:43:27,2023-03-16 23:01:19,0,80
2,0,prov6,2023-03-12 10:34:05,patient1,2023-03-10 20:43:27,2023-03-16 23:01:19,0,80
3,0,prov13,2023-03-13 09:08:10,patient1,2023-03-10 20:43:27,2023-03-16 23:01:19,0,80
4,0,prov16,2023-03-16 18:06:40,patient1,2023-03-10 20:43:27,2023-03-16 23:01:19,0,80
...,...,...,...,...,...,...,...,...
828,49,prov14,2023-02-04 10:27:03,patient50,2023-02-03 01:28:16,2023-02-09 04:33:16,0,70
829,49,prov15,2023-02-05 02:34:05,patient50,2023-02-03 01:28:16,2023-02-09 04:33:16,0,70
830,49,prov20,2023-02-09 06:24:40,patient50,2023-02-03 01:28:16,2023-02-09 04:33:16,0,70
831,49,prov16,2023-02-07 02:31:07,patient50,2023-02-03 01:28:16,2023-02-09 04:33:16,0,70


In [4]:
data.merged['hf'] = 0

In [5]:
full_df = data.merged
# full_df = full_df[(full_df['arrive_date'] <= end_date) & (full_df['date'] <= end_date)]

In [6]:
full_df.dropna().count()

id                833
dr                833
date              833
patient           833
arrive_date       833
discharge_date    833
disposition       833
age               833
hf                833
dtype: int64

In [7]:
corpus = tw.TeamworkCorpus(full_df)

Preprocessing data...
Building experience edge list...
Building team edge list...


100%|██████████| 22/22 [00:00<00:00, 417.41it/s]


In [8]:
for visit_id, item in corpus.team_experience_dict.items():
    print(f"{visit_id} {item['graph']}")

18 Graph with 4 nodes and 5 edges
45 Graph with 4 nodes and 4 edges
31 Graph with 2 nodes and 1 edges
28 Graph with 0 nodes and 0 edges
23 Graph with 2 nodes and 1 edges
42 Graph with 0 nodes and 0 edges
9 Graph with 0 nodes and 0 edges
21 Graph with 2 nodes and 1 edges
1 Graph with 2 nodes and 1 edges
30 Graph with 4 nodes and 5 edges
24 Graph with 2 nodes and 1 edges
12 Graph with 3 nodes and 2 edges
29 Graph with 2 nodes and 1 edges
35 Graph with 3 nodes and 2 edges
14 Graph with 3 nodes and 3 edges
25 Graph with 0 nodes and 0 edges
46 Graph with 0 nodes and 0 edges
19 Graph with 2 nodes and 1 edges
43 Graph with 3 nodes and 3 edges
17 Graph with 3 nodes and 2 edges
37 Graph with 0 nodes and 0 edges
6 Graph with 2 nodes and 1 edges


In [14]:
corpus.team_experience_dict[18]

{'team': {'prov1', 'prov14', 'prov16', 'prov6'},
 'graph': <networkx.classes.graph.Graph at 0x7fa7838afa10>,
 'dx_graph': <networkx.classes.graph.Graph at 0x7fa78370ae50>,
 'edgelist':    source  target  weight
 0  prov16   prov6       5
 1  prov14   prov6       1
 2  prov14  prov16       1
 3  prov14   prov6       1
 4  prov14  prov16       1
 5   prov1   prov6       2
 6   prov1  prov16       1,
 'dx_edgelist': Empty DataFrame
 Columns: [source, target, weight]
 Index: []}

In [20]:
# full_df[full_df['id'] == 18]
# corpus.edge_df[corpus.edge_df['id'] == 18]
subset = corpus.edge_df[(corpus.edge_df['dr_x'].isin(['prov16','prov14','prov1'])) & 
(corpus.edge_df['dr_y'].isin(['prov6','prov16']))]

In [21]:
subset.count()

id                     28
arrive_date            28
date                   28
dr_x                   28
hf                     28
norm_admission_date    28
norm_note_date         28
is_in_team_x           28
dr_y                   28
is_in_team_y           28
edge                   28
is_in_team             28
is_after_delta         28
dtype: int64

In [22]:
subset.arrive_date.min()

Timestamp('2023-01-08 10:14:18')

In [25]:
subset.id.unique()

array([27,  5, 49,  2,  0, 18, 23,  3, 35, 25, 19, 33,  6, 48])

In [26]:
subset = full_df[full_df['id'].isin([27,  5, 49,  2,  0, 18, 23,  3, 35, 25, 19, 33,  6, 48])]

In [27]:
subset.date.max()

Timestamp('2023-12-25 12:58:22')

In [32]:
subset

,id,dr,date,patient,arrive_date,discharge_date,disposition,age,hf
460,27,prov13,2023-01-12 01:26:21,patient28,2023-01-08 10:14:18,2023-01-14 19:19:29,0,70,0
457,27,prov6,2023-01-12 05:01:54,patient28,2023-01-08 10:14:18,2023-01-14 19:19:29,0,70,0
456,27,prov1,2023-01-13 21:43:26,patient28,2023-01-08 10:14:18,2023-01-14 19:19:29,0,70,0
455,27,prov17,2023-01-11 06:01:58,patient28,2023-01-08 10:14:18,2023-01-14 19:19:29,0,70,0
454,27,prov13,2023-01-15 04:53:07,patient28,2023-01-08 10:14:18,2023-01-14 19:19:29,0,70,0
...,...,...,...,...,...,...,...,...,...
805,48,prov1,2023-12-24 03:55:34,patient49,2023-12-18 19:27:16,2023-12-25 13:05:41,0,72,0
804,48,prov1,2023-12-24 04:33:13,patient49,2023-12-18 19:27:16,2023-12-25 13:05:41,0,72,0
803,48,prov1,2023-12-22 01:41:07,patient49,2023-12-18 19:27:16,2023-12-25 13:05:41,0,72,0
801,48,prov1,2023-12-21 22:45:33,patient49,2023-12-18 19:27:16,2023-12-25 13:05:41,0,72,0


In [45]:
import pandas as pd

def duplicate_with_date_shift(df, date_cols, interval, num_copies):
    # Create an empty DataFrame to store the results
    result_df = pd.DataFrame()
    
    # Iterate over the number of copies
    for i in range(num_copies):
        # Create a copy of the original DataFrame
        temp_df = df.copy()
        
        # Shift the date columns by (i * interval)
        for col in date_cols:
            temp_df[col] = temp_df[col] + (i * interval)

        temp_df['id'] = temp_df['id'] * (i + 1)
        
        # Append the modified DataFrame to the result DataFrame
        result_df = pd.concat([result_df, temp_df], ignore_index=True)
    
    return result_df

# Example usage:
df = pd.DataFrame({'date1': pd.date_range(start='2023-01-01', periods=5, freq='D'),
                   'date2': pd.date_range(start='2023-01-01', periods=5, freq='D')})
date_cols = ['date1', 'date2']
interval = pd.Timedelta(days=7)
num_copies = 3
new_df = duplicate_with_date_shift(df, date_cols, interval, num_copies)
print(new_df)


KeyError: 'id'

In [46]:
df = pd.read_csv('../data/sample_notes.csv')
df['date'] = pd.to_datetime(df['date'])
df['arrive_date'] = pd.to_datetime(df['arrive_date'])
df['discharge_date'] = pd.to_datetime(df['discharge_date'])

In [47]:
interval = pd.Timedelta(days=10)
num_copies = 10
duplicate_with_date_shift(df, ['arrive_date','date','discharge_date'], interval, num_copies)

,age,arrive_date,date,discharge_date,disposition,dr,hf,id,patient
0,75,2019-01-01 00:00:00,2019-01-01 19:15:00,2019-01-01,1,Brad Palmer,True,0,patient1
1,68,2019-01-24 00:00:00,2019-01-24 10:19:00,2019-01-24,0,Albert Romero,False,1,patient2
2,68,2019-01-24 00:00:00,2019-01-24 17:09:00,2019-01-24,0,Margie Meyer,False,1,patient2
3,68,2019-01-24 00:00:00,2019-01-24 16:48:00,2019-01-24,0,Evan Frazier,False,1,patient2
4,71,2019-02-14 00:00:00,2019-02-14 20:58:00,2019-02-14,0,Albert Romero,True,2,patient3
...,...,...,...,...,...,...,...,...,...
115,66,2019-07-14 19:15:00,2019-07-14 20:19:00,2019-07-14,0,Margie Meyer,True,60,patient4
116,66,2019-07-14 19:15:00,2019-07-15 04:43:00,2019-07-14,0,Evan Frazier,True,60,patient4
117,66,2019-07-14 19:15:00,2019-07-17 21:23:00,2019-07-14,0,Myrtle George,True,60,patient4
118,66,2019-07-14 19:15:00,2019-07-15 07:00:00,2019-07-14,0,Victoria Washington,True,60,patient4


In [29]:
subset_corpus = tw.TeamworkCorpus(subset)

Preprocessing data...
Building experience edge list...
Building team edge list...


Building Edge Dictionary: 100%|██████████| 186/186 [00:00<00:00, 68493.46it/s]


100%|██████████| 6/6 [00:00<00:00, 581.45it/s]


In [31]:
for visit_id, item in subset_corpus.team_experience_dict.items():
    print(item['graph'])

Graph with 4 nodes and 5 edges
Graph with 2 nodes and 1 edges
Graph with 2 nodes and 1 edges
Graph with 0 nodes and 0 edges
Graph with 0 nodes and 0 edges
Graph with 0 nodes and 0 edges


In [9]:
import pandas as pd
from datetime import timedelta

def iterate_chunks(df, datetime_col, start_date, end_date, lookback_window, target_window):
    current_start = start_date
    chunks = [] 
    while (current_start + lookback_window + target_window) < end_date:
        current_end = current_start + lookback_window + target_window
        buffered_start = current_start - timedelta(days=2)
        buffered_end = current_end + timedelta(days=2)
        chunk = df[(df[datetime_col] >= buffered_start) & (df[datetime_col] < buffered_end)]
        
        if not chunk.empty:
            # print(f"Chunk from {current_start} to {current_end}")
            # print(f"Min date: {chunk[datetime_col].min()}")
            # print(f"Max date: {chunk[datetime_col].max()}")
            chunks.append(chunk)
        
        current_start += target_window

    return chunks


In [10]:
lookback_window = timedelta(days=90)
target_window = timedelta(days=10)
full_df.sort_values('arrive_date', inplace=True)
chunks = iterate_chunks(full_df, 'arrive_date', start_date, end_date, lookback_window, target_window)

In [11]:
full_df.arrive_date.min()

Timestamp('2023-01-08 10:14:18')

In [12]:
chunks[-1]

,id,dr,date,patient,arrive_date,discharge_date,disposition,age,hf
200,12,prov7,2023-09-21 00:30:08,patient13,2023-09-16 14:19:18,2023-09-23 11:46:00,0,80,0
201,12,prov7,2023-09-21 03:34:50,patient13,2023-09-16 14:19:18,2023-09-23 11:46:00,0,80,0
202,12,prov19,2023-09-18 11:24:05,patient13,2023-09-16 14:19:18,2023-09-23 11:46:00,0,80,0
203,12,prov8,2023-09-18 16:05:45,patient13,2023-09-16 14:19:18,2023-09-23 11:46:00,0,80,0
205,12,prov11,2023-09-23 11:12:46,patient13,2023-09-16 14:19:18,2023-09-23 11:46:00,0,80,0
...,...,...,...,...,...,...,...,...,...
805,48,prov1,2023-12-24 03:55:34,patient49,2023-12-18 19:27:16,2023-12-25 13:05:41,0,72,0
804,48,prov1,2023-12-24 04:33:13,patient49,2023-12-18 19:27:16,2023-12-25 13:05:41,0,72,0
803,48,prov1,2023-12-22 01:41:07,patient49,2023-12-18 19:27:16,2023-12-25 13:05:41,0,72,0
801,48,prov1,2023-12-21 22:45:33,patient49,2023-12-18 19:27:16,2023-12-25 13:05:41,0,72,0


In [13]:
corpuses = dict()
for i, chunk in enumerate(chunks): 
    print(chunk.arrive_date.min())
    print(chunk.date.max())
    corpus_chunk = tw.TeamworkCorpus(chunk)
    corpuses[i] = corpus_chunk

2023-01-08 10:14:18
2023-04-01 18:25:27
Preprocessing data...
Building experience edge list...
Building team edge list...


Building Edge Dictionary: 100%|██████████| 143/143 [00:00<00:00, 24830.70it/s]
Building Team Dictionary: 1it [00:00, 832.70it/s]
  0%|          | 0/1 [00:00<?, ?it/s]

<class 'numpy.float64'>
nan [(np.float64(nan), (np.float64(nan), np.float64(nan)), np.float64(nan))]
np.float64(nan)


UnboundLocalError: cannot access local variable 'team' where it is not associated with a value